In [ ]:
import os
import rasterio
import pandas as pd

def check_and_load_rasters(factor_dir, win_dir):
    # 1. 获取所有tif文件路径
    factor_files = [f for f in os.listdir(factor_dir) if f.endswith('.tif')]
    PV_files = [f for f in os.listdir(win_dir) if f.endswith('.tif')]
    
    all_raster_paths = {}
    for f in factor_files:
        all_raster_paths[f.replace('.tif', '')] = os.path.join(factor_dir, f)
    for f in PV_files:
        all_raster_paths['TARGET_PV_' + f.replace('.tif', '')] = os.path.join(win_dir, f)

    print(f"--- 找到 {len(factor_files)} 个驱动力因子和 {len(PV_files)} 个目标变量 ---")
    
    # 2. 读取并检查空间参数
    meta_reports = []
    reference_meta = None
    mismatch_found = False

    for name, path in all_raster_paths.items():
        with rasterio.open(path) as src:
            meta = {
                'Name': name,
                'CRS': src.crs.to_string() if src.crs else "None",
                'Width': src.width,
                'Height': src.height,
                'Bounds': src.bounds,
                'Res': src.res,
                'NoData': src.nodata
            }
            meta_reports.append(meta)
            
            # 以第一个文件为基准进行对齐检查
            if reference_meta is None:
                reference_meta = meta
                ref_name = name
            else:
                # 检查关键参数是否一致
                if (meta['CRS'] != reference_meta['CRS'] or 
                    meta['Width'] != reference_meta['Width'] or 
                    meta['Height'] != reference_meta['Height'] or
                    meta['Res'] != reference_meta['Res']):
                    print(f"⚠️ 警告: [{name}] 与基准 [{ref_name}] 空间参数不匹配！")
                    mismatch_found = True

    # 3. 打印元数据表格
    df_meta = pd.DataFrame(meta_reports)
    
    print("\n--- 所有指标名称及地理参数概览 ---")
    print(df_meta[['Name', 'CRS', 'Width', 'Height', 'Res', 'NoData']])
    
    if not mismatch_found:
        print("\n✅ 检查通过：所有栅格数据已完美对齐。")
    else:
        print("\n❌ 检查失败：部分数据存在重投影或裁剪差异，请在建模前处理。")
        
    return all_raster_paths, df_meta

# --- 运行部分 ---
# 请确保文件夹路径正确
factor_folder = 'Factor'
PV_folder = 'PV'

paths, meta_summary = check_and_load_rasters(factor_folder, PV_folder)

In [ ]:
import numpy as np
import rasterio
import os
from rasterio.warp import reproject, Resampling

def align_and_export_all(paths, output_dir, reference_key='DEM'):
    # 1. 获取基准参数
    with rasterio.open(paths[reference_key]) as ref:
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height
        ref_profile = ref.profile.copy()
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    out_meta = ref_profile.copy()
    out_meta.update({"driver": "GTiff", "dtype": 'float32', "count": 1, "compress": 'lzw'})

    all_data_flattened = {}
    
    print(f"--- 开始对齐并导出至 {output_dir} ---")

    for name, path in paths.items():
        with rasterio.open(path) as src:
            # 准备输出矩阵
            aligned_data = np.zeros((ref_height, ref_width), dtype=np.float32)
            
            # 执行重投影/重采样
            reproject(
                source=rasterio.band(src, 1),
                destination=aligned_data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=Resampling.bilinear
            )
            
            # 保存到本地
            output_path = os.path.join(output_dir, f"{name}_aligned.tif")
            with rasterio.open(output_path, "w", **out_meta) as dest:
                dest.write(aligned_data, 1)
            
            # 处理 NoData 并存入内存供后续分析
            nodata_val = src.nodata
            temp_data = aligned_data.copy()
            if nodata_val is not None:
                temp_data[temp_data == nodata_val] = np.nan
            
            all_data_flattened[name] = temp_data.flatten()
            print(f"已完成: {name}")

    return all_data_flattened, ref_profile

# 执行整合操作 (paths 是第一步得到的路径字典)
data_matrix_fixed, final_profile = align_and_export_all(paths, "Aligned_Factors", reference_key='DEM')

# 再次生成用于建模的 DataFrame
import pandas as pd
df_final = pd.DataFrame(data_matrix_fixed).dropna().reset_index(drop=True)

print("\n--- 导出与对齐全部完成 ---")
print(f"有效像元数: {len(df_final)}")
print(df_final['PV'].value_counts())